# 05 · Selección del baseline denso

Síntesis final para la defensa: tabla comparativa de finalistas, figura principal, decisión de
baseline y limitaciones. Consume el benchmark más reciente; **no** ejecuta retrieval ni genera
embeddings.

La decisión se justifica con la métrica primaria **ParentnDCG@10** y su IC 95 % (bootstrap pareado,
semilla fija), más los controles y el coste en CPU (notebook 03).

In [ ]:
import json
from pathlib import Path

import pandas as pd

BENCH_ROOT = Path("data/processed/reports/dense/benchmarks")
runs = sorted(p for p in BENCH_ROOT.glob("*") if (p / "report.json").is_file())
if not runs:
    raise FileNotFoundError(
        f"No hay benchmarks en {BENCH_ROOT}. Ejecuta benchmark_dense_models.py (Gate C requerido)."
    )
run = runs[-1]
report = json.loads((run / "report.json").read_text(encoding="utf-8"))
summary = pd.DataFrame(
    [
        {
            "bundle_id": b["bundle_id"],
            "n_queries": b["n_queries"],
            "ParentnDCG@10": b["primary_ci"]["mean"],
            "ci_low": b["primary_ci"]["ci_low"],
            "ci_high": b["primary_ci"]["ci_high"],
        }
        for b in report.get("bundles", [])
    ]
).sort_values("ParentnDCG@10", ascending=False)
summary

In [ ]:
metrics = pd.read_csv(run / "metrics.csv")
controls = [
    c
    for c in [
        "model_alias",
        "view",
        "ParentnDCG@10",
        "ParentRecall@5",
        "EvidenceRecall@5",
        "ParentHit@1",
    ]
    if c in metrics.columns
]
metrics[controls].sort_values("ParentnDCG@10", ascending=False)  # noqa

## Decisión y limitaciones

- **Baseline elegido**: (completar con el bundle ganador y su `benchmark_id`/fingerprints).
- **Justificación**: ParentnDCG@10 + controles + coste CPU; las diferencias deben superar el IC 95 %.
- **Limitaciones**: dataset de evaluación pequeño (revisión jurídica manual), CPU-only, dense-only
  (sin BM25/híbrido/reranking, fuera de alcance de esta fase).

Exporta la figura final a `thesis/figures/` para la memoria.